import json
import base64
from google.cloud import vision
import google.generativeai as genai
from typing import Dict, List, Any
import os
from pathlib import Path
from dotenv import load_dotenv

In [37]:
load_dotenv()

api_key = os.getenv('GEMINI_API_KEY')

genai.configure(api_key=api_key)

In [ ]:
class VisionToGeminiProcessor:
    def __init__(self):
        self.gemini_model = genai.GenerativeModel()
    
    def extract_text_from_vision_results(self, vision_response) -> Dict[str, Any]:
        """
        Extract structured text data from Google Vision API response
        """
        extracted_data = {
            'full_text': '',
            'structured_text': [],
            'nutrition_keywords': [],
            'ingredient_keywords': [],
            'regulatory_keywords': []
        }
        
        if hasattr(vision_response, 'text_annotations') and vision_response.text_annotations:
            # Get full text (first annotation contains all text)
            extracted_data['full_text'] = vision_response.text_annotations[0].description
            
            # Get individual text elements with bounding boxes
            for annotation in vision_response.text_annotations[1:]:  # Skip first (full text)
                text_element = {
                    'text': annotation.description,
                    'confidence': getattr(annotation, 'confidence', 0),
                    'bounding_box': {
                        'vertices': [(vertex.x, vertex.y) for vertex in annotation.bounding_poly.vertices]
                    }
                }
                extracted_data['structured_text'].append(text_element)
        
        # Extract nutrition-related keywords
        nutrition_keywords = ['calories', 'fat', 'sodium', 'protein', 'carbohyrates', 'sugar', 'fiber', 'vitamin']
        ingredient_keywords = ['ingredients', 'contains', 'may contain', 'allergens']
        regulatory_keywords = ['FDA', 'USDA', 'organic', 'natural', 'gluten-free', 'non-GMO']
        
        text_lower = extracted_data['full_text'].lower()
        
        for keyword in nutrition_keywords:
            if keyword in text_lower:
                extracted_data['nutrition_keywords'].append(keyword)
        
        for keyword in ingredient_keywords:
            if keyword in text_lower:
                extracted_data['ingredient_keywords'].append(keyword)
                
        for keyword in regulatory_keywords:
            if keyword in text_lower:
                extracted_data['regulatory_keywords'].append(keyword)
        
        return extracted_data
    
    def process_with_gemini(self, extracted_text: str, user_question: str) -> Dict[str, Any]:
        """
        Process extracted text with Gemini API
        """
        prompt = f"""
        You are a food regulation expert. I have extracted text from a food package using OCR.
        
        EXTRACTED TEXT:
        {extracted_text}
        
        USER QUESTION: {user_question}
        
        Please analyze this food information and answer the question. Consider:
        1. Nutrition facts and daily values
        2. Ingredient list and allergen information
        3. Compliance requirements
        4. Health claims and labeling standards
        
        - Direct answer to the question
        - Additional insights
        
        """
        
        try:
            response = self.gemini_model.generate_content(prompt)
            
            # Try to parse as JSON, fallback to text response
            try:
                parsed_response = json.loads(response.text)
                return parsed_response
            except json.JSONDecodeError:
                return {
                    'answer': response.text,
                    'raw_response': True
                }
                
        except Exception as e:
            return {
                'error': str(e),
                'answer': 'Unable to process the question at this time.'
            }

# Example usage functions

def load_vision_results_from_notebook(notebook_path: str, cell_output_index: int = 0):
    """
    Load Vision API results from Jupyter notebook output
    """
    # If you have saved results as JSON
    try:
        with open('extraction_results.json', 'r') as f:
            vision_data = json.load(f)
        return vision_data
    except FileNotFoundError:
        print("Result not found")
        return None

def process_food_package_question(vision_results_file: str, user_question: str):
   
    processor = VisionToGeminiProcessor()
    
    # Load Vision API results
    with open(vision_results_file, 'r') as f:
        vision_data = json.load(f)
    
    # Extract text from Vision results
    extracted_text = vision_data.get('full_text', '')
    
    # Process with Gemini
    result = processor.process_with_gemini(extracted_text, user_question)
    
    return result

# Example: Integration with your existing notebook
def integrate_with_notebook():

    processor = VisionToGeminiProcessor()
    
    # Load the saved results
    with open('vision_results.json', 'r') as f:
        vision_data = json.load(f)
    
    extracted_text = vision_data.get('full_text', '')
    
    # Example questions
    questions = [
        "What are the nutrition facts for this product?",
        "Does this product comply with FDA labeling requirements?",
        "What allergens are present in this product?",
        "How many calories per serving?",
        "Are there any artificial preservatives?"
    ]
    
    results = []
    for question in questions:
        result = processor.process_with_gemini(extracted_text, question)
        results.append({
            'question': question,
            'response': result
        })
    
    return results

# Web API integration (for your Node.js backend)
def create_api_endpoint_handler():
    """
    Example handler for your Node.js API
    """
    api_code = '''
    // Node.js API endpoint
    const { GoogleGenerativeAI } = require("@google/generative-ai");
    
    const genAI = new GoogleGenerativeAI(process.env.GEMINI_API_KEY);
    const model = genAI.getGenerativeModel({ model: "gemini-pro" });
    
    app.post('/api/analyze-food-package', async (req, res) => {
        try {
            const { extractedText, question } = req.body;
            
            const prompt = `
                Analyze this food package text and answer: ${question}
                
                EXTRACTED TEXT: ${extractedText}
                
                Provide structured analysis focusing on nutrition, ingredients, and regulations.
            `;
            
            const result = await model.generateContent(prompt);
            const response = await result.response;
            
            res.json({
                success: true,
                answer: response.text(),
                timestamp: new Date().toISOString()
            });
            
        } catch (error) {
            res.status(500).json({
                success: false,
                error: error.message
            });
        }
    });
    '''
    
    return api_code

In [ ]:
if __name__ == "__main__":
    try:
        result = process_food_package_question(
            'extraction_results.json', 
            'Is this food consider healthy? This is a corned beef'
        )
        print(json.dumps(result, indent=2))
    except Exception as e:
        print(f"Error: {e}")

{
  "answer": "**Direct Answer to the Question:**\n\nNo, this corned beef is not considered a healthy food choice based on the provided nutrition information.\n\n**Additional Insights:**\n\n**1. Nutrition Facts and Daily Values:**\n\n* **High in Fat:**  The corned beef contains 15.5g of total fat and 9.2g of saturated fat per 100g serving.  Saturated fat is linked to increased cholesterol levels and heart disease.  The percentage daily values (%DV) aren't fully shown, but the 22% for saturated fat indicates it's a significant portion of the recommended daily intake.  This is a major concern.\n\n* **High in Sodium:** With 445mg of sodium per 100g serving (and implied 1.1g per serving, showing 46% DV), this corned beef is very high in sodium. Excessive sodium intake contributes to high blood pressure and other health problems.\n\n* **Low in Fiber and other micronutrients:** The lack of dietary fiber and minimal carbohydrates suggests a lack of essential nutrients.  The product is primari

: 